In [1]:
!pip install pandas numpy matplotlib seaborn plotly sqlalchemy requests scipy jupyter


   ------------- -------------------------- 2/6 [lark]
   -------------------- ------------------- 3/6 [fqdn]
   ---------------------------------------- 6/6 [isoduration]



In [5]:
import pandas as pd
import numpy as np
import requests
import os
import glob
import json

### Create project folder structure:project/ ■■■ data/raw/ data/processed/ ■■■ notebooks/■■■ sql/ ■■■ dashboard/

In [6]:
folders = [
    "data/raw",
    "data/processed",
    "notebooks",
    "sql",
    "dashboard",
    "reports"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [7]:
csv_files = glob.glob("data/raw/*.csv")

print("Total CSV Files:", len(csv_files))

for file in csv_files:
    print(os.path.basename(file))
    

Total CSV Files: 10
01_fund_master.csv
02_nav_history.csv
03_aum_by_fund_house.csv
04_monthly_sip_inflows.csv
05_category_inflows.csv
06_industry_folio_count.csv
07_scheme_performance.csv
08_investor_transactions.csv
09_portfolio_holdings.csv
10_benchmark_indices.csv


### Load all 10 CSV datasets usingpandas, print shape/dtypes/head for each

In [8]:
for file in os.listdir("data/raw"):
    
    if file.endswith(".csv"):
        
        print("\n", "="*50)
        print("File:", file)

        df = pd.read_csv("data/raw/" + file)

        print("Shape:", df.shape)

        print("\nData Types:")
        print(df.dtypes)

        print("\nFirst 5 Rows:")
        print(df.head())


File: 01_fund_master.csv
Shape: (40, 15)

Data Types:
amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object
dtype: object

First 5 Rows:
   amfi_code       fund_house                                   scheme_name  \
0     119551  SBI Mutual Fund     SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund      SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund    SBI Small Cap Fund - Regular Plan - Growth   
3     119599  SBI Mutual Fund     SBI Small Cap Fund - Direct Plan - Growth   
4     119120  SBI Mutual Fund  SBI Magnum Gilt Fund - Regular Plan - Gr

### Fetch live NAV from mfapi.in: GEThttps://api.mfapi.in/mf/125497 (HDFCTop 100) Parse JSON response and save as raw CSV

In [9]:
url = "https://api.mfapi.in/mf/125497"

response = requests.get(url)

data = response.json()

df = pd.DataFrame(data["data"])

print(df.head())

         date        nav
0  19-06-2026  202.07610
1  18-06-2026  200.95650
2  17-06-2026  199.83020
3  16-06-2026  198.61520
4  15-06-2026  198.03200


In [10]:
df.to_csv("data/raw/HDFC_Top100.csv",index=False)
print("saved")

saved


### Fetch NAV for 5 schemes: SBIBluechip (119551), ICICI Bluechip (120503), Nippon Large Cap (118632), Axis Bluechip (119092),
### Kotak Bluechip (120841)

In [11]:
schemes = {
    "119551": "SBI_Bluechip",
    "120503": "ICICI_Bluechip",
    "118632": "Nippon_LargeCap",
    "119092": "Axis_Bluechip",
    "120841": "Kotak_Bluechip"
}

for code, name in schemes.items():

    url = "https://api.mfapi.in/mf/" + code

    response = requests.get(url)

    data = response.json()

    df = pd.DataFrame(data["data"])

    df.to_csv("data/raw/" + name + ".csv", index=False)

    print(name, "Saved")

SBI_Bluechip Saved
ICICI_Bluechip Saved
Nippon_LargeCap Saved
Axis_Bluechip Saved
Kotak_Bluechip Saved


In [13]:
fund = pd.read_csv("data/raw/01_fund_master.csv")

print(fund.columns)

Index(['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category',
       'plan', 'launch_date', 'benchmark', 'expense_ratio_pct',
       'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager',
       'risk_category', 'sebi_category_code'],
      dtype='object')


### Understand fund master: print uniquefund houses, categories,sub-categories, risk grades

In [17]:
fund_master = pd.read_csv(r'data/raw/01_fund_master.csv')
fund_master.head()

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [18]:
fund_master["fund_house"].unique()

array(['SBI Mutual Fund', 'HDFC Mutual Fund', 'ICICI Prudential MF',
       'Nippon India MF', 'Kotak Mahindra MF', 'Axis Mutual Fund',
       'Aditya Birla Sun Life MF', 'UTI Mutual Fund', 'Mirae Asset MF',
       'DSP Mutual Fund'], dtype=object)

In [20]:
fund_master["category"].unique()

array(['Equity', 'Debt'], dtype=object)

In [19]:
fund_master["sub_category"].unique()

array(['Large Cap', 'Small Cap', 'Gilt', 'Mid Cap', 'Short Duration',
       'Value', 'Liquid', 'Index/ETF', 'Flexi Cap', 'Index',
       'Large & Mid Cap', 'ELSS'], dtype=object)

In [21]:
fund_master["risk_category"].unique()

array(['Moderate', 'Very High', 'Low', 'High', 'Moderately High'],
      dtype=object)

In [22]:
print("Number of Fund Houses:", fund_master["fund_house"].nunique())
print("Number of Categories:", fund_master["category"].nunique())
print("Number of Sub Categories:", fund_master["sub_category"].nunique())
print("Number of Risk Categories:", fund_master["risk_category"].nunique())

Number of Fund Houses: 10
Number of Categories: 2
Number of Sub Categories: 12
Number of Risk Categories: 5


### Task 7 (Validate AMFI Codes)

In [23]:
nav_history = pd.read_csv("data/raw/02_nav_history.csv")

print(nav_history.columns)

Index(['amfi_code', 'date', 'nav'], dtype='object')


In [24]:
fund_master = pd.read_csv("data/raw/01_fund_master.csv")
nav_history = pd.read_csv("data/raw/02_nav_history.csv")

# AMFI codes in each dataset
fund_codes = set(fund_master["amfi_code"])
nav_codes = set(nav_history["amfi_code"])

# Codes present in fund_master but missing in nav_history
missing_codes = fund_codes - nav_codes

print("Total AMFI Codes in Fund Master:", len(fund_codes))
print("Total AMFI Codes in NAV History:", len(nav_codes))
print("Missing AMFI Codes:", len(missing_codes))

print("\nMissing Codes:")
print(missing_codes)

Total AMFI Codes in Fund Master: 40
Total AMFI Codes in NAV History: 40
Missing AMFI Codes: 0

Missing Codes:
set()


In [26]:
print(" DATA QUALITY SUMMARY ")
print(f"Total Fund Master Codes : {len(fund_codes)}")
print(f"Total NAV History Codes : {len(nav_codes)}")
print(f"Missing Codes           : {len(missing_codes)}")

if len(missing_codes) == 0:
    print("\n All AMFI codes are present in NAV History.")
else:
    print("\n Some AMFI codes are missing in NAV History.")

 DATA QUALITY SUMMARY 
Total Fund Master Codes : 40
Total NAV History Codes : 40
Missing Codes           : 0

 All AMFI codes are present in NAV History.
